In [1]:

from pathlib import Path
import sys
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve().parent
SRC = ROOT / "src"

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from ras.core import ras_sharpe
from ras.pbo_comparison import prepare_pbo_data_for_ras
from backtest_overfitting.main import run_cscv
from backtest_overfitting.overfit_statistics import compute_pbo

# The notebook is located inside the notebooks/ folder
ROOT = Path.cwd().parent

# Simulation settings from the existing PBO experiment
experiment = "demo_experiment"
seed = 42

# Directory containing the simulated strategy returns
returns_dir = (
    ROOT
    / "data"
    / "processed"
    / experiment
    / f"seed_{seed}"
    / "returns"
)

print("\nChecking files:")
print("No-signal data exists:",
      (returns_dir / "no_signal.parquet").exists())
print("Seasonal data exists:",
      (returns_dir / "seasonal.parquet").exists())



Checking files:
No-signal data exists: True
Seasonal data exists: True


In [2]:

import pandas as pd
import numpy as np

# Load the existing PBO strategy-return matrices
no_signal = pd.read_parquet(
    returns_dir / "no_signal.parquet"
)

seasonal = pd.read_parquet(
    returns_dir / "seasonal.parquet"
)

# Examine their structure
print("No-signal shape:", no_signal.shape)
print("Seasonal shape:", seasonal.shape)

print("\nFirst five rows and five strategies:")
display(no_signal.iloc[:5, :5])

print("\nMissing values:")
print("No signal:", no_signal.isna().sum().sum())
print("Seasonal:", seasonal.isna().sum().sum())

print("\nMatching strategy columns:",
      no_signal.columns.equals(seasonal.columns))

print("Matching observation dates:",
      no_signal.index.equals(seasonal.index))


No-signal shape: (1000, 7600)
Seasonal shape: (1000, 7600)

First five rows and five strategies:


,22,23,24,25,26
date,,,,,
2000-01-03,NaN,NaN,NaN,NaN,NaN
2000-01-04,NaN,NaN,NaN,NaN,NaN
2000-01-05,NaN,NaN,NaN,NaN,NaN
2000-01-06,NaN,NaN,NaN,NaN,NaN
2000-01-10,NaN,NaN,NaN,NaN,NaN



Missing values:
No signal: 3470135
Seasonal: 3479814

Matching strategy columns: True
Matching observation dates: True



## Data Preparation: Missing Values and Strategy Activity

The CSCV/PBO simulation produces two strategy-return matrices with 1,000 observations and 7,600 strategies each.

Both matrices contain missing values. The strategy generator uses NaN for unrecorded P&L observations and retains explicit zero-valued observations. For this exploratory RAS comparison, we interpret unrecorded P&L as zero when constructing full-calendar strategy returns. This changes the performance convention relative to the original PBO calculation.

This distinction matters because RAS requires finite observations, and replacing missing values with zero may change the interpretation of each strategy's performance.


In [3]:

# Examine missing observations and strategy activity

for name, df in [
    ("No Signal", no_signal),
    ("Seasonal", seasonal)
]:
    print(f"\n--- {name} ---")

    valid_counts = df.notna().sum(axis=0)

    print("Strategies:", df.shape[1])
    print("Observations:", df.shape[0])
    print("Min valid observations:", valid_counts.min())
    print("Median valid observations:", valid_counts.median())
    print("Max valid observations:", valid_counts.max())

    print("Strategies with zero valid observations:",
          (valid_counts == 0).sum())

    print("Strategies with fewer than 10 valid observations:",
          (valid_counts < 10).sum())

    print("Strategies with all 1000 observations:",
          (valid_counts == len(df)).sum())

    print("\nFirst strategy sample:")
    print(df.iloc[:20, 0].to_string())



--- No Signal ---
Strategies: 7600
Observations: 1000
Min valid observations: 22
Median valid observations: 547.0
Max valid observations: 999
Strategies with zero valid observations: 0
Strategies with fewer than 10 valid observations: 0
Strategies with all 1000 observations: 0

First strategy sample:
date
2000-01-03         NaN
2000-01-04         NaN
2000-01-05         NaN
2000-01-06         NaN
2000-01-10         NaN
2000-01-11         NaN
2000-01-12         NaN
2000-01-13         NaN
2000-01-17         NaN
2000-01-18         NaN
2000-01-19         NaN
2000-01-20         NaN
2000-01-24         NaN
2000-01-25         NaN
2000-01-26         NaN
2000-01-27         NaN
2000-01-31         NaN
2000-02-01    0.000000
2000-02-02   -0.041657
2000-02-03         NaN

--- Seasonal ---
Strategies: 7600
Observations: 1000
Min valid observations: 22
Median valid observations: 538.5
Max valid observations: 999
Strategies with zero valid observations: 0
Strategies with fewer than 10 valid observation

In [4]:

# Examine recorded trading P&L without modifying the data

for name, df in [
    ("No Signal", no_signal),
    ("Seasonal", seasonal)
]:
    values = df.to_numpy(dtype=float)
    recorded = values[np.isfinite(values)]

    print(f"\n--- {name} ---")
    print("Recorded observations:", len(recorded))
    print("Mean recorded P&L:", np.mean(recorded))
    print("Std recorded P&L:", np.std(recorded))
    print("Fraction of recorded P&L equal to zero:",
          np.mean(recorded == 0))
    print("Minimum P&L:", np.min(recorded))
    print("Maximum P&L:", np.max(recorded))



--- No Signal ---
Recorded observations: 4129865
Mean recorded P&L: 0.0024996506981882663
Std recorded P&L: 0.9769699416029798
Fraction of recorded P&L equal to zero: 0.0875125458096088
Minimum P&L: -3.75205874306533
Maximum P&L: 3.75205874306533

--- Seasonal ---
Recorded observations: 4120186
Mean recorded P&L: 0.002050692207775329
Std recorded P&L: 0.9864330676676525
Fraction of recorded P&L equal to zero: 0.08786569344199509
Minimum P&L: -3.7520587430653336
Maximum P&L: 3.7520587430653336


In [5]:

# Analyze individual strategy volatility and activity

for name, df in [
    ("No Signal", no_signal),
    ("Seasonal", seasonal)
]:
    strategy_std = df.std(axis=0, ddof=1)
    activity_rate = df.notna().mean(axis=0)

    print(f"\n--- {name} ---")

    print("Strategy volatility:")
    print(strategy_std.describe())

    print("\nStrategy observation rate:")
    print(activity_rate.describe())

    print("\nZero-volatility strategies:",
          (strategy_std == 0).sum())

    print("Missing or undefined volatility:",
          strategy_std.isna().sum())



--- No Signal ---
Strategy volatility:
count    7600.000000
mean        0.949858
std         0.082389
min         0.504526
25%         0.930613
50%         0.972415
75%         0.999785
max         1.103712
dtype: float64

Strategy observation rate:
count    7600.000000
mean        0.543403
std         0.284566
min         0.022000
25%         0.286000
50%         0.547000
75%         0.794000
max         0.999000
dtype: float64

Zero-volatility strategies: 0
Missing or undefined volatility: 0

--- Seasonal ---
Strategy volatility:
count    7600.000000
mean        0.958022
std         0.084781
min         0.503329
25%         0.933846
50%         0.983076
75%         1.010261
max         1.125329
dtype: float64

Strategy observation rate:
count    7600.000000
mean        0.542130
std         0.284415
min         0.022000
25%         0.286000
50%         0.538500
75%         0.793000
max         0.999000
dtype: float64

Zero-volatility strategies: 0
Missing or undefined volatility: 0


In [6]:

# Compare recorded-observation and full-calendar volatility

for name, df in [
    ("No Signal", no_signal),
    ("Seasonal", seasonal)
]:
    recorded_std = df.std(axis=0, ddof=1)

    # Create a separate candidate matrix; leave original unchanged
    calendar_pnl = df.fillna(0.0)
    calendar_std = calendar_pnl.std(axis=0, ddof=1)

    print(f"\n--- {name} ---")
    print("Mean recorded-observation std:", recorded_std.mean())
    print("Mean full-calendar std:", calendar_std.mean())
    print("Minimum full-calendar std:", calendar_std.min())
    print("Zero full-calendar std:", (calendar_std == 0).sum())
    print("Median volatility ratio:",
          (calendar_std / recorded_std).median())



--- No Signal ---
Mean recorded-observation std: 0.9498582260349464
Mean full-calendar std: 0.6813988269289905
Minimum full-calendar std: 0.07380941891285961
Zero full-calendar std: 0
Median volatility ratio: 0.7398800727505097

--- Seasonal ---
Mean recorded-observation std: 0.9580219167526229
Mean full-calendar std: 0.6868115736952318
Minimum full-calendar std: 0.073468671123305
Zero full-calendar std: 0
Median volatility ratio: 0.734238360583586




## Constructing the RAS Input Matrix

RAS requires a finite matrix of volatility-normalized strategy observations:

$$
X \in \mathbb{R}^{T \times N}
$$

where:

- $T$ is the number of simulated observations.
- $N$ is the number of trading strategies.
- $X_{t,j}$ is the normalized P&L of strategy $j$ at time $t$.

### 1. Handling Missing Observations

The original PBO strategy-return matrices contain missing values because some trading strategies do not record P&L on every date.

For our exploratory RAS experiment, we interpret these unrecorded observations as zero P&L to obtain a full-calendar matrix.

$$
r_{t,j} =
\begin{cases}
\text{Recorded P\&L}_{t,j}, & \text{if observed},\\
0, & \text{if unrecorded}.
\end{cases}
$$

### 2. Volatility Normalization

Each strategy is normalized by its full-calendar sample standard deviation:

$$
\hat{\sigma}_j =
\sqrt{
\frac{1}{T-1}
\sum_{t=1}^{T}
(r_{t,j}-\bar{r}_j)^2
}
$$

The normalized observations are:

$$
X_{t,j} = \frac{r_{t,j}}{\hat{\sigma}_j}
$$

We do not subtract the sample mean during normalization because we want to preserve each strategy's empirical performance.

### 3. Interpretation and Limitations

Both RAS and PBO use the same underlying simulated market data and strategy configurations.

However, RAS uses a full-calendar, volatility-normalized matrix, while the existing PBO implementation evaluates recorded P&L observations.

The volatility estimates use the full sample, so this is a retrospective comparison rather than a prospective trading procedure.

Furthermore, the applicability of the theoretical RAS confidence bound to these trading returns has not yet been established.


In [7]:



X_no_signal = prepare_pbo_data_for_ras(no_signal)
X_seasonal = prepare_pbo_data_for_ras(seasonal)

print("No-signal RAS matrix:", X_no_signal.shape)
print("Seasonal RAS matrix:", X_seasonal.shape)

print("\nNo-signal values finite:", np.isfinite(X_no_signal).all())
print("Seasonal values finite:", np.isfinite(X_seasonal).all())

print("\nMean strategy volatility after normalization:")
print("No Signal:", np.std(X_no_signal, axis=0, ddof=1).mean())
print("Seasonal:", np.std(X_seasonal, axis=0, ddof=1).mean())


No-signal RAS matrix: (1000, 7600)
Seasonal RAS matrix: (1000, 7600)

No-signal values finite: True
Seasonal values finite: True

Mean strategy volatility after normalization:
No Signal: 1.0
Seasonal: 1.0



## First RAS Experiment: 100 Trading Strategies

### Objective

We begin by applying Rademacher Anti-Serum (RAS) to a subset of 100 strategies from the same simulated data used for PBO.

We consider two scenarios:

1. **No Signal:** Strategies evaluated on a random walk without a predictable component.
2. **Seasonal Signal:** Strategies evaluated on a price series with a seasonal component.

Both scenarios use the same strategy configurations and the same underlying random shocks.

### 1. Rademacher Complexity

RAS estimates the complexity of the strategy universe using random signs:

$$
\widehat{\mathcal{R}}_T
=
\frac{1}{B}
\sum_{b=1}^{B}
\max_{1 \leq j \leq N}
\left(
\frac{1}{T}
\sum_{t=1}^{T}
\epsilon_t^{(b)} X_{t,j}
\right)
$$

where:

- $T = 1000$ is the number of observations.
- $N = 100$ is the number of strategies.
- $B = 1000$ is the number of Monte Carlo draws.
- $\epsilon_t^{(b)}$ takes values $-1$ or $+1$ with equal probability.
- $X_{t,j}$ is the normalized P&L for strategy $j$ at time $t$.

### 2. RAS Performance Adjustment

The implementation uses a data-snooping penalty:

$$
D = 2\widehat{\mathcal{R}}_T
$$

and an estimation-error term:

$$
E =
3\sqrt{\frac{2\log(2/\delta)}{T}}
+
\sqrt{\frac{2\log(2N/\delta)}{T}}
$$

The total haircut is:

$$
H = D + E
$$

The adjusted lower-bound statistic for strategy $j$ is:

$$
L_j = \widehat{SR}_j - H
$$

where:

$$
\widehat{SR}_j =
\frac{1}{T}\sum_{t=1}^{T}X_{t,j}
$$

In our experiment, $\delta = 0.1$.

### 3. Interpretation

A larger Rademacher complexity indicates a larger estimated data-snooping penalty.

The adjusted statistic measures how much empirical performance remains after subtracting the RAS haircut.

This initial calculation is exploratory. We have not established that the theoretical RAS confidence guarantee applies to the dependent trading P&L generated by our strategies.

The first 100 strategy columns are used only as a computational test, not as a representative sample of the full strategy universe.


In [8]:

import sys
import numpy as np

# Import the RAS implementation from our repository
src_path = str(ROOT / "src")

if src_path not in sys.path:
    sys.path.insert(0, src_path)

from ras.core import ras_sharpe
from ras.pbo_comparison import prepare_pbo_data_for_ras

# Experiment parameters
N_test = 100
B = 1000
delta = 0.1
seed = 42

# Same 100 strategy configurations in both scenarios
X_no_test = X_no_signal[:, :N_test]
X_seasonal_test = X_seasonal[:, :N_test]

# Run your partner's RAS implementation
ras_no_signal = ras_sharpe(
    X_no_test,
    delta=delta,
    B=B,
    seed=seed
)

ras_seasonal = ras_sharpe(
    X_seasonal_test,
    delta=delta,
    B=B,
    seed=seed
)

# Display results
for name, result in [
    ("No Signal", ras_no_signal),
    ("Seasonal Signal", ras_seasonal)
]:
    print(f"\n--- {name} ---")

    print("Rademacher complexity:",
          round(result["rademacher_complexity"], 6))

    print("Data-snooping penalty:",
          round(result["data_snooping"], 6))

    print("Estimation error:",
          round(result["estimation_error"], 6))

    print("Total haircut:",
          round(result["haircut"], 6))

    print("Best empirical Sharpe:",
          round(np.max(result["empirical_sharpe"]), 6))

    print("Best adjusted lower bound:",
          round(np.max(result["lower_bound"]), 6))

    print("Strategies with positive lower bound:",
          np.sum(result["lower_bound"] > 0))



--- No Signal ---
Rademacher complexity: 0.043095
Data-snooping penalty: 0.086189
Estimation error: 0.355509
Total haircut: 0.441698
Best empirical Sharpe: 0.068513
Best adjusted lower bound: -0.373185
Strategies with positive lower bound: 0

--- Seasonal Signal ---
Rademacher complexity: 0.043418
Data-snooping penalty: 0.086836
Estimation error: 0.355509
Total haircut: 0.442345
Best empirical Sharpe: 0.118416
Best adjusted lower bound: -0.323929
Strategies with positive lower bound: 0


In [9]:
# Load PBO summaries for the same simulation
from pathlib import Path
import pandas as pd

pbo_dir = (
    ROOT
    / "results"
    / "replication"
    / experiment
    / f"seed_{seed}"
    / "tables"
)

for name in ["no_signal", "seasonal"]:
    summary_path = pbo_dir / name / "sharpe" / "summary.csv"

    print(f"\n--- {name.upper()} ---")

    if summary_path.exists():
        summary = pd.read_csv(summary_path)
        print(summary.to_string(index=False))
    else:
        print("PBO summary not found:", summary_path)


--- NO_SIGNAL ---
metric      pbo  degradation_slope  degradation_intercept  degradation_r2  degradation_defined  probability_of_loss  first_order_dominance_vs_split_mean  second_order_dominance_vs_split_mean  oos_score_benchmark  probability_below_oos_benchmark  S    T  n_strategies  n_excluded_strategies          units
sharpe 0.702381          -0.672009               0.897049        0.115516                 True              0.68254                                False                                 False                  0.0                          0.68254 10 1000          7600                      0 price_unit_pnl

--- SEASONAL ---
metric      pbo  degradation_slope  degradation_intercept  degradation_r2  degradation_defined  probability_of_loss  first_order_dominance_vs_split_mean  second_order_dominance_vs_split_mean  oos_score_benchmark  probability_below_oos_benchmark  S    T  n_strategies  n_excluded_strategies          units
sharpe 0.051587          -0.635627              


## Matched 100-Strategy Comparison

The saved PBO results use 7,600 strategies, whereas the initial RAS experiment uses only 100.

To make the comparison more consistent, we rerun CSCV/PBO on the same 100 strategy configurations used for RAS.

Both methods therefore evaluate the same simulated market scenarios and strategy subset.

PBO retains its original recorded-P&L convention, while RAS uses full-calendar normalized P&L. Consequently, the comparison matches the strategy universe but does not yet fully harmonize the performance definitions.

This preliminary experiment assesses whether the two methods distinguish between the no-signal and seasonal scenarios.


In [10]:

import scipy

from backtest_overfitting.main import run_cscv
from backtest_overfitting.overfit_statistics import compute_pbo
import numpy as np

# Use the same 100 strategy configurations as RAS
N_test = 100
S = 10

# Run CSCV on the original strategy P&L matrices
pbo_no_100 = run_cscv(
    no_signal.iloc[:, :N_test],
    S=S
)

pbo_seasonal_100 = run_cscv(
    seasonal.iloc[:, :N_test],
    S=S
)

# Compute PBO
pbo_no_value = compute_pbo(pbo_no_100)
pbo_seasonal_value = compute_pbo(pbo_seasonal_100)

# Display results
print("--- Matched 100-Strategy Comparison ---")
print(f"No-signal PBO: {pbo_no_value:.2%}")
print(f"Seasonal PBO: {pbo_seasonal_value:.2%}")

print("\nRAS adjusted lower bounds:")
print(f"No signal: {np.max(ras_no_signal['lower_bound']):.4f}")
print(f"Seasonal: {np.max(ras_seasonal['lower_bound']):.4f}")

print("\nNumber of CSCV splits:")
print("No signal:", len(pbo_no_100))
print("Seasonal:", len(pbo_seasonal_100))

from backtest_overfitting.main import run_cscv
from backtest_overfitting.overfit_statistics import compute_pbo
import numpy as np



CSCV: 100%|██████████| 252/252 [00:00<00:00, 1613.08split/s]

--- Matched 100-Strategy Comparison ---
No-signal PBO: 37.70%
Seasonal PBO: 0.00%

RAS adjusted lower bounds:
No signal: -0.3732
Seasonal: -0.3239

Number of CSCV splits:
No signal: 252
Seasonal: 252



## Full-Universe Comparison: 7,600 Strategies

The initial experiment compared PBO and RAS using the first 100 trading strategies.

We now extend the analysis to all 7,600 strategies generated by the original simulation.

The objective is to investigate whether increasing the number of candidate strategies changes the RAS complexity penalty and the resulting adjusted performance.

The full-universe PBO results have already been computed using CSCV with 10 blocks.

For RAS, we retain the same simulated price paths, strategy configurations, and volatility-normalization convention used in our initial experiment.

This is still an exploratory comparison because the theoretical applicability of the RAS bound to these strategy P&L observations has not been established.


In [11]:

# Full-universe RAS experiment: computational requirements

T, N = X_no_signal.shape
B = 1000

print("Observations (T):", T)
print("Strategies (N):", N)
print("Monte Carlo draws (B):", B)

print("\nApproximate memory requirements:")
print(f"Strategy matrix: {X_no_signal.nbytes / 1024**2:.2f} MiB")
print(f"Random signs: {B * T * 8 / 1024**2:.2f} MiB")
print(f"RAS score matrix: {B * N * 8 / 1024**2:.2f} MiB")


Observations (T): 1000
Strategies (N): 7600
Monte Carlo draws (B): 1000

Approximate memory requirements:
Strategy matrix: 57.98 MiB
Random signs: 7.63 MiB
RAS score matrix: 57.98 MiB



## Full-Universe RAS Evaluation

We apply Rademacher Anti-Serum (RAS) to all 7,600 trading strategies generated by the original PBO simulation.

The experiment uses:

- $T = 1000$ observations.
- $N = 7600$ trading strategies.
- $B = 1000$ Monte Carlo Rademacher draws.
- Confidence parameter $\delta = 0.10$.
- Random seed $42$ for reproducibility.

We evaluate two scenarios using the same underlying simulated random shocks:

1. No-signal random walk.
2. Random walk containing a seasonal signal.

The estimated complexity penalty is:

$$
D = 2\widehat{\mathcal{R}}_T
$$

The total adjustment is:

$$
H = D + E
$$

where $E$ is the estimation-error term.

The adjusted performance statistic for strategy $j$ is:

$$
L_j = \widehat{SR}_j - H
$$

We compare the maximum adjusted performance and the number of strategies with positive adjusted bounds across both scenarios.

These calculations are exploratory because the theoretical assumptions required for the RAS lower-bound guarantee have not yet been established for these dependent trading P&L observations.


In [12]:

from ras.core import ras_sharpe
import numpy as np
import pandas as pd

# Full strategy universe
B = 1000
delta = 0.10
seed = 42

ras_full_results = {}

for name, X in [
    ("No Signal", X_no_signal),
    ("Seasonal", X_seasonal)
]:
    print(f"\nRunning full RAS: {name}")

    result = ras_sharpe(
        X,
        delta=delta,
        B=B,
        seed=seed
    )

    ras_full_results[name] = result

    print("Rademacher complexity:",
          round(result["rademacher_complexity"], 6))
    print("Data-snooping penalty:",
          round(result["data_snooping"], 6))
    print("Estimation error:",
          round(result["estimation_error"], 6))
    print("Total haircut:",
          round(result["haircut"], 6))
    print("Best empirical Sharpe:",
          round(np.max(result["empirical_sharpe"]), 6))
    print("Best adjusted lower bound:",
          round(np.max(result["lower_bound"]), 6))
    print("Positive adjusted bounds:",
          int(np.sum(result["lower_bound"] > 0)))



Running full RAS: No Signal
Rademacher complexity: 0.082724
Data-snooping penalty: 0.165447
Estimation error: 0.386691
Total haircut: 0.552139
Best empirical Sharpe: 0.074454
Best adjusted lower bound: -0.477685
Positive adjusted bounds: 0

Running full RAS: Seasonal
Rademacher complexity: 0.082728
Data-snooping penalty: 0.165455
Estimation error: 0.386691
Total haircut: 0.552147
Best empirical Sharpe: 0.129051
Best adjusted lower bound: -0.423095
Positive adjusted bounds: 0



## Effect of Strategy Universe Size on RAS

We examine how increasing the number of candidate strategies from $N=100$ to $N=7600$ changes RAS.

The RAS haircut is defined as:

$$
H(N) = 2\widehat{\mathcal{R}}_T(N) + E(N)
$$

Increasing the strategy universe expands the number of candidate configurations that can be selected based on observed performance.

We investigate whether this produces:

1. Higher estimated Rademacher complexity.
2. A larger total adjustment.
3. Lower adjusted performance estimates.

The comparison uses the same $T=1000$ observations and identical simulation seeds.

Because the smaller experiment uses the first 100 strategy columns rather than a randomly selected subset, these results are descriptive and should not be interpreted as a general scaling law.


In [13]:

import pandas as pd
import numpy as np

experiments = [
    ("No Signal", 100, ras_no_signal),
    ("No Signal", 7600, ras_full_results["No Signal"]),
    ("Seasonal", 100, ras_seasonal),
    ("Seasonal", 7600, ras_full_results["Seasonal"])
]

rows = []

for scenario, n, result in experiments:
    rows.append({
        "Scenario": scenario,
        "Strategies": n,
        "Rademacher Complexity": result["rademacher_complexity"],
        "Estimation Error": result["estimation_error"],
        "Total Haircut": result["haircut"],
        "Best Empirical Sharpe": np.max(result["empirical_sharpe"]),
        "Best Adjusted Bound": np.max(result["lower_bound"]),
        "Positive Bounds": int(np.sum(result["lower_bound"] > 0))
    })

ras_comparison = pd.DataFrame(rows)

display(ras_comparison.round(4))


,Scenario,Strategies,Rademacher Complexity,Estimation Error,Total Haircut,Best Empirical Sharpe,Best Adjusted Bound,Positive Bounds
0,No Signal,100,0.0431,0.3555,0.4417,0.0685,-0.3732,0
1,No Signal,7600,0.0827,0.3867,0.5521,0.0745,-0.4777,0
2,Seasonal,100,0.0434,0.3555,0.4423,0.1184,-0.3239,0
3,Seasonal,7600,0.0827,0.3867,0.5521,0.1291,-0.4231,0



## Preliminary Findings

### Effect of Strategy Universe Size

Increasing the strategy universe from 100 to 7,600 strategies increases the estimated Rademacher complexity and the total RAS haircut in both simulated scenarios.

For the no-signal simulation, Rademacher complexity increases from approximately 0.0431 to 0.0827.

For the seasonal simulation, complexity increases from approximately 0.0434 to 0.0827.

The stronger complexity penalty results in more negative adjusted lower-bound statistics, despite slight increases in the highest empirical Sharpe ratios.

### Comparison with PBO

Using all 7,600 strategies, CSCV produces:

- No-signal PBO: 70.24%.
- Seasonal PBO: 5.16%.

This indicates substantially greater in-sample selection instability in the no-signal scenario.

RAS also produces a higher best empirical Sharpe statistic for the seasonal scenario, but neither simulation achieves a positive adjusted lower-bound statistic.

### Interpretation

The preliminary results suggest that PBO distinguishes the two simulated scenarios more clearly in this experiment, whereas RAS applies substantial complexity and estimation-error penalties to both.

However, the methods answer different statistical questions, use different preprocessing conventions, and have different theoretical assumptions.

Therefore, these results do not establish that one method is more accurate or generally superior.

### Next Research Objective

Repeat the comparison across independently generated simulation seeds to evaluate the stability of the findings.

Before drawing formal inferential conclusions, verify the theoretical assumptions underlying the RAS bound and conduct sensitivity analysis of the treatment of unrecorded trading P&L.
